# RAG from Scratch — Live Demo
**AI Engineering · Module 1 · Lecture 3**

We'll build a complete RAG pipeline from scratch:
1. Load and parse a document
2. Chunk it intelligently
3. Embed chunks using the Nebius API
4. Store in a vector database (ChromaDB)
5. Retrieve relevant chunks for a query
6. Generate a grounded answer

> **Run on Google Colab — no GPU required for this demo**

## 0. Setup

In [1]:
# Install dependencies
# ingnore errors in this cell

!pip install openai chromadb pypdf tiktoken -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
import os
import json
import textwrap
from pathlib import Path
from typing import List, Dict, Tuple

from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

# ── Nebius API config ─────────────────────────────────────────────────────
NEBIUS_API_KEY = 'YOUR_NEBIUS_API_KEY'   # ← paste your token
NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1/"

EMBED_MODEL  = "BAAI/bge-en-icl"            # Nebius embedding model
LLM_MODEL    = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # generation model

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print("Client ready ✓")

Client ready ✓


## 1. Load a Document

We'll use a short PDF as our knowledge base. You can swap this for any PDF you like.

In [3]:
# Download a sample PDF (IEA Global Energy Review — public document)
import urllib.request

PDF_URL  = "https://iea.blob.core.windows.net/assets/5b169aa1-bc88-4c96-b828-aaa50406ba80/GlobalEnergyReview2025.pdf"
PDF_PATH = "sample.pdf"

try:
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    from pypdf import PdfReader
    reader = PdfReader(PDF_PATH)
    raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    print(f"PDF loaded: {len(reader.pages)} pages, {len(raw_text):,} characters")
except Exception as e:
    raise(f"PDF unavailable ({e})")

print("\nFirst 500 chars:")
print(raw_text[:500])

PDF loaded: 43 pages, 84,190 characters

First 500 chars:
Global Energy 
Review 2025
The IEA examines the full 
spectrum 
of energy issues 
including oil, gas and 
coal supply and 
demand, renewable 
energy technologies, 
electricity markets, 
energy efficiency, 
access to energy, 
demand side 
management and much 
more. Through its work, 
the IEA advocates 
policies that will enhance 
the reliability, 
affordability and 
sustainability of energy 
in its  
32 Member countries,   
13 Association countries 
and beyond.
This publication and any map 
inclu


## 2. Chunk the Document

We'll use **recursive character chunking** — split on paragraphs first, then sentences if a chunk is still too large. This is more 'logical' than fixed-size splitting.

In [4]:
def chunk_text(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80
) -> List[str]:
    """
    Simple recursive chunker:
    1. Split on double newlines (paragraphs)
    2. If a paragraph is too large, split on single newlines
    3. If still too large, split on sentence boundaries
    4. Apply overlap between consecutive chunks
    """
    # Clean up whitespace
    text = " ".join(text.split())

    # Split into candidate sentences (~words)
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= chunk_size:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            # Start new chunk with overlap from previous
            if chunks and overlap > 0:
                prev_words = chunks[-1].split()
                overlap_text = " ".join(prev_words[-overlap//5:])
                current = (overlap_text + " " + sentence).strip()
            else:
                current = sentence

    if current:
        chunks.append(current)

    return [c for c in chunks if len(c.strip()) > 30]


chunks = chunk_text(raw_text, chunk_size=400, overlap=80)

print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c) for c in chunks) // len(chunks)} chars")
print("\n--- Chunk 0 ---")
print(chunks[0])
print("\n--- Chunk 1 ---")
print(chunks[1] if len(chunks) > 1 else "(only one chunk)")

Total chunks: 356
Avg chunk size: 341 chars

--- Chunk 0 ---
Global Energy Review 2025 The IEA examines the full spectrum of energy issues including oil, gas and coal supply and demand, renewable energy technologies, electricity markets, energy efficiency, access to energy, demand side management and much more.

--- Chunk 1 ---
renewable energy technologies, electricity markets, energy efficiency, access to energy, demand side management and much more. Through its work, the IEA advocates policies that will enhance the reliability, affordability and sustainability of energy in its 32 Member countries, 13 Association countries and beyond.


## 3. Embed Chunks & Build Vector Store

We embed each chunk using the Nebius embedding API, then store everything in ChromaDB (in-memory).

In [5]:
def embed_texts(texts: List[str], model: str = EMBED_MODEL, verbose=False) -> List[List[float]]:
    """Embed a list of texts using the Nebius API."""
    # Nebius accepts batches; use chunks of 32 to be safe
    all_embeddings = []
    batch_size = 32

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        if verbose:
            print(f"\r  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks",  end='', flush=True)

    print()
    return all_embeddings


print("Embedding chunks...")
embeddings = embed_texts(chunks, verbose=True)
print(f"Done. Each embedding has {len(embeddings[0])} dimensions.")

Embedding chunks...
  Embedded 356/356 chunks
Done. Each embedding has 4096 dimensions.


In [6]:
# Store in ChromaDB (in-memory)
chroma_client = chromadb.Client()

# Delete collection if it already exists (idempotent for re-runs)
try:
    chroma_client.delete_collection("rag_demo")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_demo",
    metadata={"hnsw:space": "cosine"}   # cosine similarity
)

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"Vector store ready: {collection.count()} chunks stored ✓")

Vector store ready: 356 chunks stored ✓


## 4. Retrieve — Semantic Search

Given a query, embed it and find the most similar chunks using cosine similarity.

In [7]:
def retrieve(query: str, top_k: int = 3) -> List[Dict]:
    """Embed query and retrieve top_k most similar chunks."""
    query_embedding = embed_texts([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "distances"]
    )

    retrieved = []
    for i, (doc, dist) in enumerate(zip(
        results["documents"][0],
        results["distances"][0]
    )):
        retrieved.append({
            "rank": i + 1,
            "text": doc,
            "similarity": round(1 - dist, 4)  # ChromaDB returns distance, we flip to similarity
        })

    return retrieved


# Test retrieval
query = "How much did renewable energy capacity grow in 2024?"
results = retrieve(query, top_k=3)

print(f"Query: {query}\n")
for r in results:
    print(f"[Rank {r['rank']} | Similarity: {r['similarity']}]")
    print(textwrap.fill(r['text'], width=90))
    print()


Query: How much did renewable energy capacity grow in 2024?

[Rank 1 | Similarity: 0.7396]
on nuclear (23% of the total), with lower shares for natural gas (16%) and coal (11%).
Technology: Solar PV and wind In 2024, global annual renewable capacity additions surged
by an estimated 25% to around 700 GW – marking the 22nd consecutive year that renewables
have set new records for expansion.

[Rank 2 | Similarity: 0.7381]
remainder. Solar PV additions in 2024 rose by almost 30% year -over-year, totalling about
550 GW. With this growth , installed solar PV capacity worldwide reached an estimated 2.2
terawatts (TW). Annual wind additions remained stable at around 120 GW. Together, solar PV
and wind accounted for 95% of overall renewable capacity growth in 2024.

[Rank 3 | Similarity: 0.7317]
Together, they contributed 40% of total generation for the first time, with renewables
alone supplying 32%. New renewables installations hit record levels for the 22nd
consecutive year, with around 700

## 5. Generate — Grounded Answer

Pass the retrieved chunks + the original query to the LLM. The prompt explicitly tells the model to answer only from the provided context.

In [8]:
def build_prompt(query: str, retrieved_chunks: List[Dict]) -> str:
    """Build the RAG prompt with runtime data enhancement."""
    context_parts = []
    for chunk in retrieved_chunks:
        # Runtime data enhancement: tell the model the rank and similarity
        context_parts.append(
            f"[Source {chunk['rank']} | Relevance: {chunk['similarity']}]\n{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    return f"""Answer the following question using ONLY the provided context.
If the context does not contain enough information, say so clearly.
Do not make up information.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""


def generate(query: str, top_k: int = 3) -> str:
    """Full RAG pipeline: retrieve then generate."""
    # Retrieve
    retrieved = retrieve(query, top_k=top_k)

    # Build prompt
    prompt = build_prompt(query, retrieved)

    # Generate
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a precise assistant. Answer questions based strictly on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


# Try it!
answer = generate(query)
print(f"Q: {query}\n")
print(f"A: {answer}")


Q: How much did renewable energy capacity grow in 2024?

A: According to the context, renewable energy capacity grew by an estimated 25% in 2024, with a total of around 700 GW added.


## 6. Try It Yourself

Run the full pipeline on different questions:

In [9]:
questions = [
    "What happened to CO2 emissions in 2024?",
    "How much clean energy investment was there compared to fossil fuels?",
    "What is the role of electric vehicles in oil demand?",
    "What are the main uses of coal globally?",   # ← partially answerable
    "Who won the 2024 US presidential election?",  # ← NOT in the context
]

for q in questions:
    print(f"{'='*70}")
    print(f"Q: {q}")
    print(f"A: {generate(q)}")
    print()

Q: What happened to CO2 emissions in 2024?

A: According to Source 1, total energy-related CO2 emissions increased by 0.8% in 2024, hitting an all-time high of 37.8 Gt CO2.

Q: How much clean energy investment was there compared to fossil fuels?

A: There is not enough information in the provided context to answer the question about clean energy investment compared to fossil fuels. The context only discusses electricity generation and global energy supply growth, but does not mention investment.

Q: What is the role of electric vehicles in oil demand?

A: According to the context, electric vehicles are mentioned as "substitutes" that are contributing to the growth of oil demand slowing down. Specifically, it is stated that "the increasing growth of substitutes like electric vehicles" is one of the factors that contributed to the slower oil demand growth in 2024.

Q: What are the main uses of coal globally?

A: The main uses of coal globally are not explicitly stated in the provided con

---
## Summary

We built a complete RAG pipeline:

| Step | What we did |
|------|-------------|
| **Pre-process** | Extracted text from PDF |
| **Chunk** | Sentence-aware chunking with overlap |
| **Embed** | Nebius embedding API (BGE model) |
| **Store** | ChromaDB in-memory vector store |
| **Retrieve** | Cosine similarity search |
| **Generate** | Nebius LLM with grounding prompt |

**Key observations:**
- Relavent chunks are retrived
- The last question (US election) was answered correctly as "not in context" — grounding works!

**Next:** In the evaluation notebook, we'll measure *how good* this pipeline actually is.